In [69]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input

In [70]:
train_dir = "/kaggle/input/chest-xray-pneumonia/chest_xray/train"
val_dir   = "/kaggle/input/chest-xray-pneumonia/chest_xray/val"
test_dir  = "/kaggle/input/chest-xray-pneumonia/chest_xray/test"

In [71]:
! mkdir -p -/.kaggle
! cp kaggle.json ~/.kaggle/

mkdir: invalid option -- '/'
Try 'mkdir --help' for more information.


In [74]:
!pip install kaggle --quiet #install kaggle

In [75]:
#upload kaggle.json file
from google.colab import files
files.upload()

Saving kaggle.json to kaggle (1).json


{'kaggle (1).json': b'{"username":"swaliha1410","key":"1531d47c7e06cb06068965d7e5e4bc0b"} '}

In [76]:
#create kaggle direct
!mkdir -p ~/.kaggle

In [77]:
#copy kaggle.json
!cp kaggle.json ~/.kaggle/

In [78]:
ls -ltr ~/.kaggle

total 4
-rw------- 1 root root 68 May 31 09:18 kaggle.json


In [79]:
!chmod 600 ~/.kaggle/kaggle.json

In [80]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [ ]:
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia

In [ ]:
!unzip chest-xray-pneumonia.zip

In [ ]:
import os

print(os.listdir("chest_xray"))

In [ ]:
import tensorflow as tf

train_data = tf.keras.preprocessing.image_dataset_from_directory(
    "chest_xray/train",
    image_size=(128,128),
    batch_size=32
)

test_data = tf.keras.preprocessing.image_dataset_from_directory(
    "chest_xray/test",
    image_size=(128,128),
    batch_size=32
)

In [ ]:
print(train_data.class_names)

In [ ]:
import matplotlib.pyplot as plt

for images, labels in train_data.take(1):

    plt.imshow(images[0].numpy().astype("uint8"))

    plt.title(train_data.class_names[labels[0]])

    plt.axis("off")

    plt.show()

In [ ]:
from tensorflow.keras import layers, models

model = models.Sequential([

    layers.Rescaling(1./255, input_shape=(128,128,3)),

    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),

    layers.Dense(128, activation='relu'),

    layers.Dense(1, activation='sigmoid')
])

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
test_loss, test_acc = model.evaluate(test_data)

print("Accuracy:", test_acc)

In [ ]:
model.save("pneumonia_model.h5")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img("11.jpg", target_size=(128,128))

img_array = image.img_to_array(img)

img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

if prediction[0][0] > 0.5:
    print("PNEUMONIA")
else:
    print("NORMAL")

In [ ]:
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image

# Load saved model
model = tf.keras.models.load_model('pneumonia_model.h5')

# App title
st.title("🩺 Pneumonia Detection System")
st.write("Upload a Chest X-Ray image to detect Pneumonia")

# Upload image
uploaded_file = st.file_uploader(
    "Choose X-Ray image...",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:

    # Show uploaded image
    image = Image.open(uploaded_file)
    st.image(image, caption='Uploaded X-Ray', width=300)

    # Preprocess image
    img = image.resize((150, 150))
    img = img.convert('RGB')
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Predict
    prediction = model.predict(img_array)
    confidence = prediction[0][0]

    # Show result
    st.subheader("🔍 Result:")

    if confidence > 0.5:
        st.error(f"⚠️ PNEUMONIA DETECTED")
        st.write(f"Confidence: {confidence*100:.2f}%")
    else:
        st.success(f"✅ NORMAL - No Pneumonia")
        st.write(f"Confidence: {(1-confidence)*100:.2f}%")

In [ ]:
import os
print(os.listdir())

In [ ]:
!pip install streamlit

In [ ]:
!pip install pyngrok

In [ ]:
from google.colab import files
files.upload()

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
import os
print(os.listdir())

In [ ]:
!pip install streamlit pyngrok

In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

In [ ]:
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
!cloudflared tunnel --url http://localhost:8501

In [ ]:
%%writefile app.py
import streamlit as st
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from PIL import Image
import numpy as np
st.set_page_config(
    page_title="Pneumonia Detection",
    page_icon="🩺",
    layout="wide"
)

st.markdown("""
<style>
.main {
    padding-top: 2rem;
}
.hero {
    background: linear-gradient(135deg, #1e3c72, #2a5298);
    padding: 2rem;
    border-radius: 15px;
    color: white;
    text-align: center;
}
.feature-box {
    background-color: #f0f2f6;
    padding: 20px;
    border-radius: 10px;
    margin-top: 10px;
}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="hero">
    <h1>🩺 Pneumonia Detection System</h1>
    <h3>Deep Learning Based Chest X-Ray Analysis</h3>
    <p>Upload a chest X-ray image and detect Pneumonia using AI.</p>
</div>
""", unsafe_allow_html=True)

st.write("")
st.write("")

col1, col2, col3 = st.columns(3)

with col1:
    st.info("📊 AI Powered Analysis")

with col2:
    st.success("⚡ Fast Prediction")

with col3:
    st.warning("🏥 Medical Imaging")

st.markdown("----")
# Load model
@st.cache_resource
def load_my_model():
    return load_model("pneumonia_model.h5")

model = load_my_model()

st.title("🩺 Pneumonia Detection System")

uploaded_file = st.file_uploader(
    "Upload Chest X-ray",
    type=["jpg", "jpeg", "png"]
)

if uploaded_file is not None:

    img = Image.open(uploaded_file)

    st.image(
        img,
        caption="Uploaded Chest X-ray",
        use_container_width=True
    )

    if st.button("Predict"):

        # Preprocess image
        img = img.resize((128, 128))
        img_array = np.array(img)

        # If image is grayscale, convert to 3 channels
        if len(img_array.shape) == 2:
            img_array = np.stack((img_array,) * 3, axis=-1)

        img_array = np.expand_dims(img_array, axis=0)

        # Predict
        prediction = model.predict(img_array)

        score = prediction[0][0]

        st.subheader("Prediction Result")

        if score > 0.5:
            st.error("⚠️ PNEUMONIA DETECTED")
            st.write(f"Confidence Score: {score:.2f}")
        else:
            st.success("✅ NORMAL")
            st.write(f"Confidence Score: {1-score:.2f}")